# Photonic QGAN with `PhotonicGenerator`

This notebook demonstrates the MerLin `PhotonicGenerator` API on a compact photonic QGAN-style workflow. It mirrors the structure used in the reproduced photonic QGAN implementation: repeated quantum generator heads produce image patches, the patches are adapted into image tensors, and Adam optimizes both the generator and a classical discriminator.

![Photonic QGAN overview](../../_static/reproduced_papers/photonicQGAN.png)

The full paper reproduction lives in the reproduced-papers project. The example here is deliberately small so it can run as documentation: it checks the model wiring, the occupancy readout used by the photonic QGAN mapping, and one short Adam training loop.


In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset

import merlin as ML


_ = torch.manual_seed(7)


## Build repeated photonic generator heads

Each head is a `QuantumLayer` returning Fock-space probabilities with `occupancy_readout=True`. The readout collapses count-resolved Fock states to occupied/unoccupied bins before the `ImageAdapter` crops or pads each head into an image patch.


In [2]:
def make_generator_head(latent_dim: int = 2) -> ML.QuantumLayer:
    builder = ML.CircuitBuilder(n_modes=3)
    builder.add_entangling_layer(trainable=True, name="var_0")
    builder.add_angle_encoding(modes=list(range(latent_dim)), name="enc")
    builder.add_entangling_layer(trainable=True, name="var_1")

    return ML.QuantumLayer(
        input_size=latent_dim,
        builder=builder,
        input_state=[1, 1, 0],
        measurement_strategy=ML.MeasurementStrategy.probs(
            computation_space=ML.ComputationSpace.FOCK,
            occupancy_readout=True,
        ),
    )


latent_dim = 2
image_shape = (1, 4, 4)

generator = ML.PhotonicGenerator(
    layers=make_generator_head(latent_dim),
    count=2,
    output_adapter=ML.ImageAdapter(
        shape=image_shape,
        headwise=True,
        normalize_patches=True,
    ),
)

z = generator.sample_latent(batch_size=4)
measurements = generator.measure(z)
generated = generator(z)

print(f"heads: {len(generator)}")
print(f"latent shape: {tuple(z.shape)}")
print(f"per-head output widths: {[output.shape[1] for output in measurements.outputs]}")
print(f"image batch shape: {tuple(generated.shape)}")
print(f"value range: [{generated.min().item():.3f}, {generated.max().item():.3f}]")


heads: 2
latent shape: (4, 2)
per-head output widths: [6, 6]
image batch shape: (4, 1, 4, 4)
value range: [0.000, 1.000]


## Prepare a tiny image batch

The original reproduction trains on image data. For a fast documentation run, we use a tiny subset of scikit-learn digits and average-pool the 8x8 images down to 4x4 so the quantum generator stays small.


In [3]:
digits = load_digits()
zero_images = torch.tensor(digits.images[digits.target == 0], dtype=torch.float32)[:8]
zero_images = zero_images / zero_images.max()
small_images = F.avg_pool2d(zero_images.unsqueeze(1), kernel_size=2)
real_vectors = small_images.reshape(small_images.shape[0], -1)
loader = DataLoader(
    TensorDataset(real_vectors, torch.zeros(len(real_vectors))),
    batch_size=4,
    shuffle=False,
)

print(f"training batch shape: {tuple(real_vectors.shape)}")


training batch shape: (8, 16)


## Train with Adam

This is the same optimizer ownership model as the reproduced implementation: MerLin exposes generator parameters as PyTorch parameters, `loss.backward()` populates their gradients, and Adam performs the updates.


In [4]:
discriminator = nn.Sequential(
    nn.Linear(16, 16),
    nn.LeakyReLU(0.2),
    nn.Linear(16, 1),
)

criterion = nn.BCEWithLogitsLoss()
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=0.01, betas=(0.5, 0.999))
optimizer_g = torch.optim.Adam(generator.parameters(), lr=0.01, betas=(0.5, 0.999))

loss_history: list[tuple[float, float]] = []

for step, (real_batch, _) in enumerate(loader):
    if step >= 2:
        break

    batch_size = real_batch.shape[0]
    real_labels = torch.full((batch_size,), 0.9)
    fake_labels = torch.zeros(batch_size)
    generator_labels = torch.full((batch_size,), 0.9)

    noise_d = torch.normal(0.0, 2 * math.pi, (batch_size, latent_dim))
    fake_for_d = generator(noise_d).reshape(batch_size, -1).detach()

    optimizer_d.zero_grad()
    d_loss = criterion(discriminator(real_batch).view(-1), real_labels)
    d_loss = d_loss + criterion(discriminator(fake_for_d).view(-1), fake_labels)
    d_loss.backward()
    optimizer_d.step()

    noise_g = torch.normal(0.0, 2 * math.pi, (batch_size, latent_dim))
    optimizer_g.zero_grad()
    fake_for_g = generator(noise_g).reshape(batch_size, -1)
    g_loss = criterion(discriminator(fake_for_g).view(-1), generator_labels)
    g_loss.backward()
    optimizer_g.step()

    loss_history.append((float(d_loss.detach()), float(g_loss.detach())))

has_generator_gradient = any(
    param.grad is not None and param.grad.abs().sum() > 0
    for param in generator.parameters()
)

print(f"loss history: {loss_history}")
print(f"generator received gradients: {has_generator_gradient}")


loss history: [(1.4189794063568115, 0.6906531453132629), (1.3897066116333008, 0.6990799903869629)]
generator received gradients: True


## What this validates

The notebook exercises the public MerLin path needed by the photonic QGAN reproduction:

- repeated independent generator heads via `PhotonicGenerator(..., count=...)`;
- Fock-space probability readout with `occupancy_readout=True`;
- headwise image adaptation and patch normalization;
- generator and discriminator updates with standard PyTorch Adam optimizers.

For a direct implementation-level comparison against the reproduced-papers `PatchGenerator.dist_to_image` mapping, run `docs/run_pml282_photonic_qgan_reference_comparison.py` from the repository root when the local `external/reproduced_papers` checkout is available.
